# Gulfstream walkthrough — equities (`equity_eod`)

Same Graph **1** → Graph **2** story as the YCS notebook, on a DAX30 log-price panel,
then repeated with **kernel PCA** and **DMD**.

| Part | Dimred | Pipelines |
|------|--------|-----------|
| A | PCA | Graph 1 → Graph 2 (seeded from Graph 1) |
| B | Kernel PCA | Graph 1 → Graph 2 |
| C | DMD | Graph 1 → Graph 2 |

**Database:** `D:/data/duckdb/equity_eod_data.duckdb` · **table:** `equity_eod`

> **Windows tip:** if DuckDB says the file is locked, close DBeaver and re-run.
> Falls back to `equity_eod_data_copy.duckdb` when present.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import duckdb
import pandas as pd
import polars as pl
from plotnine import aes, geom_line, ggplot, labs, theme_bw, facet_wrap, theme

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = Path(r"D:/Code/gulfstream")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

EQ_CANDIDATES = [
    Path(r"D:/data/duckdb/equity_eod_data.duckdb"),
    Path(r"D:/data/duckdb/equity_eod_data_copy.duckdb"),
]
EQ_DB = next((p for p in EQ_CANDIDATES if p.exists()), EQ_CANDIDATES[0])
OUT_DIR = ROOT / "outputs" / "notebooks" / "equity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("EQ_DB =", EQ_DB, "exists =", EQ_DB.exists())


## 1. Schema & coverage

Columns: `Index` (date), `Stock`, OHLC, `Volume`, `EqIndex` (e.g. `DAX30`, `CAC40`, `SP500`).


In [ ]:
def open_equity() -> duckdb.DuckDBPyConnection:
    try:
        return duckdb.connect(str(EQ_DB), read_only=True)
    except Exception as exc:
        raise RuntimeError(
            f"Could not open {EQ_DB}. Close DBeaver / other DuckDB clients and retry.\n{exc}"
        ) from exc

con = open_equity()
print(con.execute("DESCRIBE equity_eod").pl())
print(
    con.execute(
        """
        SELECT EqIndex, COUNT(*) AS n, COUNT(DISTINCT Stock) AS stocks,
               MIN(Index) AS dmin, MAX(Index) AS dmax
        FROM equity_eod
        GROUP BY 1
        ORDER BY n DESC
        """
    ).pl()
)
con.close()


## 2. Build a wide close panel

**DAX30** over the GFC window so structural breaks are visible; keep `N_TICKERS` names.


In [ ]:
EQ_INDEX = "DAX30"
START, END = "2007-01-01", "2012-12-31"
N_TICKERS = 8

con = open_equity()
long = con.execute(
    f"""
    SELECT CAST(Index AS DATE) AS date,
           Stock AS ticker,
           Close AS close
    FROM equity_eod
    WHERE EqIndex = '{EQ_INDEX}'
      AND Index >= '{START}'
      AND Index <= '{END}'
    ORDER BY date, ticker
    """
).pl()
con.close()

from gulfstream.common import frames

wide = (
    long.pivot(values="close", index="date", on="ticker", aggregate_function="first")
    .sort("date")
)
wide = frames.ensure_date_column(wide)
tickers = frames.feature_columns(wide)[:N_TICKERS]
wide = wide.select(["date", *tickers]).drop_nulls()
print("wide shape:", wide.shape)
print("tickers:", tickers)
wide.head(3)


## 3. Visualize raw closes


In [ ]:
long_px = (
    wide.unpivot(index="date", on=tickers, variable_name="ticker", value_name="close")
    .to_pandas()
)
long_px["date"] = pd.to_datetime(long_px["date"])

(
    ggplot(long_px, aes("date", "close", color="ticker"))
    + geom_line(size=0.35)
    + theme_bw()
    + theme(figure_size=(11, 4))
    + labs(title=f"{EQ_INDEX} closes ({START} → {END})", x="", y="Close")
)


## 4. Feature engineering

Log prices (+ short-horizon return vols) — a dated wide polars frame ready for gulfstream.


In [ ]:
features_df = wide.with_columns([pl.col(c).log().alias(c) for c in tickers])
vol_window = 20
for c in tickers:
    r = f"{c}_ret"
    v = f"{c}_vol"
    features_df = features_df.with_columns((pl.col(c).diff()).alias(r))
    features_df = features_df.with_columns(pl.col(r).rolling_std(vol_window).alias(v))

features_df = features_df.drop_nulls()
print("features:", features_df.shape, "n_feat:", frames.n_features(features_df))
plot_cols = tickers[:3]
features_df.head(3)


## 5. Feature snapshot


In [ ]:
viz_cols = tickers[:3] + [f"{tickers[0]}_vol", f"{tickers[1]}_vol"]
viz_cols = [c for c in viz_cols if c in features_df.columns]
long_f = (
    features_df.select(["date", *viz_cols])
    .unpivot(index="date", on=viz_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long_f["date"] = pd.to_datetime(long_f["date"])

(
    ggplot(long_f, aes("date", "value", color="series"))
    + geom_line(size=0.35)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.0 * len(viz_cols)), legend_position="none")
    + labs(title="Equity features fed to gulfstream", x="", y="")
)


## 6. Shared helpers

Graph 2 **reuses** each Graph 1 run by seeding `retrain.regimes_df` from that run's
breakpoints. Equity defaults use a slightly looser MMD gate so breaks survive in this window.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream.common import frames, utils
from gulfstream.detection import time_index as bkpt_time
from gulfstream.metrics import regime_plots
from gulfstream.metrics.regime_plots import _visualize_market_regimes
from gulfstream.pipelines.hamilton.driver import run_segmentation_pair
from gulfstream.pipelines.graph2 import targeted_retrain_with_user_specified_df


def load_core_params(img_dir: Path) -> dict:
    """Graph 1 core YAML with notebook-friendly metrics toggles."""
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    """Return a deep copy configured for pca / kpca / dmd."""
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method == "dmd":
        # Keep the window modest so Graph 2 regime slices still fit.
        out["algo"]["dmd_stride"] = [5]
        out["algo"]["dmd_rolling_window"] = [20]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred for this notebook: {method}")
    return out


def seed_regimes_from_results(df: pl.DataFrame, res) -> pl.DataFrame:
    """Convert a Graph 1 SegmentResults into a Graph 2 ``regimes_df`` seed.

    Graph 2 reads ``End`` on all but the last row as *breakpoint dates*
    (see ``regimes_df_to_bkpts``), so ``End`` must be ``dates[bkpt]``, not the
    last observation of the preceding regime.
    """
    dates = frames.dates_series(df).to_list()
    n = len(dates)
    bkpts = sorted(int(b) for b in (res.bkpts or []) if 0 < int(b) < n)
    hierarchy = {
        int(k): int(v)
        for k, v in (res.hierarchy or {b: 1 for b in bkpts}).items()
    }
    rows = []
    for i, b in enumerate(bkpts):
        start_i = 0 if i == 0 else bkpts[i - 1]
        rows.append(
            {
                "Start": dates[start_i],
                "End": dates[b],
                "Regime": i,
                "Hierarchy Level of End": int(hierarchy.get(b, 1)),
            }
        )
    start_last = bkpts[-1] if bkpts else 0
    rows.append(
        {
            "Start": dates[start_last],
            "End": dates[n - 1],
            "Regime": len(bkpts),
            "Hierarchy Level of End": 0,
        }
    )
    return pl.DataFrame(rows)


def summarize_breakpoints(df: pl.DataFrame, res, label: str) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def show_regimes(df: pl.DataFrame, res, title: str, variables: list[str]) -> pl.DataFrame:
    dates = frames.dates_series(df).to_list()
    hierarchy = res.hierarchy or {b: 1 for b in res.bkpts}
    regimes_df = bkpt_time._get_regime_intervals(hierarchy, dates)
    vars_ = [c for c in variables if c in frames.feature_columns(df)][:2]
    _visualize_market_regimes(
        df,
        regimes_df,
        title=title,
        variables=vars_ or frames.feature_columns(df)[:2],
        valid_bkpts=res.bkpts,
        invalid_bkpts=res.invalid_bkpts,
        low_confidence_bkpts=list(res.low_confidence_bkpts or []),
        mode="display",
    )
    return regimes_df


def run_graph1(df: pl.DataFrame, params: dict, label: str, plot_vars: list[str]):
    """Hamilton single-pass (Graph 1 core). Returns (unprocessed, processed)."""
    print(f"=== Graph 1 · {label} · dimred={params['algo']['dimred']} ===")
    unproc, proc = run_segmentation_pair(df, params)
    summarize_breakpoints(df, proc, label)
    show_regimes(df, proc, f"Graph 1 · {label}", plot_vars)
    return unproc, proc


def run_graph2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
) -> Path:
    """Graph 2 auto-retrain seeded from a Graph 1 ``SegmentResults``."""
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    seed = seed_regimes_from_results(df, seed_res)
    print(f"=== Graph 2 · {label} · seed regimes ===")
    print(seed.to_dicts())
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "regimes_df": seed,
    }
    targeted_retrain_with_user_specified_df(df, g2)
    # Surface heatmaps written by the retrain loop
    pngs = sorted(out_dir.rglob("retrain_iteration_*.png"))
    if not pngs:
        pngs = sorted(out_dir.rglob("*.png"))[:6]
    print(f"Graph 2 artifacts under {out_dir} ({len(list(out_dir.rglob('*')))} files)")
    for p in pngs[:8]:
        print(" ", p.relative_to(out_dir))
        try:
            display(Image(filename=str(p)))
        except Exception as exc:
            print("  (could not display)", exc)
    return out_dir


print("Helpers ready: load_core_params, with_dimred, run_graph1, run_graph2")


In [ ]:
def equity_params(img_dir: Path) -> dict:
    """PCA/kPCA/DMD-ready params with equity-friendly MMD settings."""
    params = load_core_params(img_dir)
    params["metrics"]["features_to_plot"] = plot_cols
    params["algo"]["min_regime_length"] = [20]
    params["algo"]["depth"] = [2]
    params["test"]["significance_level"] = [0.2]
    params["test"]["window"] = [{"method": "user_specified", "window": 60}]
    params["test"]["sample_size"] = [{"method": "user_specified", "num_samples": 60}]
    return params


---
# Part A — PCA (baseline)


## A.1 Graph 1 (PCA)


In [ ]:
params_pca = with_dimred(equity_params(OUT_DIR / "pca"), "pca")
unproc_pca, proc_pca = run_graph1(features_df, params_pca, "PCA", plot_cols)
seed_regimes_from_results(features_df, proc_pca)


## A.2 Graph 2 (seeded from PCA Graph 1)


In [ ]:
g2_pca_dir = run_graph2(
    features_df,
    params_pca,
    proc_pca,
    OUT_DIR / "pca" / "graph2",
    "PCA",
    max_iter=3,
)


---
# Part B — Kernel PCA


## B.1 Graph 1 (kernel PCA)


In [ ]:
params_kpca = with_dimred(equity_params(OUT_DIR / "kpca"), "kpca")
unproc_kpca, proc_kpca = run_graph1(features_df, params_kpca, "kPCA", plot_cols)


## B.2 Graph 2 (seeded from kPCA Graph 1)


In [ ]:
g2_kpca_dir = run_graph2(
    features_df,
    params_kpca,
    proc_kpca,
    OUT_DIR / "kpca" / "graph2",
    "kPCA",
    max_iter=3,
)


---
# Part C — DMD


## C.1 Graph 1 (DMD)


In [ ]:
params_dmd = with_dimred(equity_params(OUT_DIR / "dmd"), "dmd")
unproc_dmd, proc_dmd = run_graph1(features_df, params_dmd, "DMD", plot_cols)


## C.2 Graph 2 (seeded from DMD Graph 1)


In [ ]:
g2_dmd_dir = run_graph2(
    features_df,
    params_dmd,
    proc_dmd,
    OUT_DIR / "dmd" / "graph2",
    "DMD",
    max_iter=3,
)


---
# Comparison


In [ ]:
dates = frames.dates_series(features_df).to_list()

def bkpt_table(label, res):
    return {
        "dimred": label,
        "n_bkpts": len(res.bkpts),
        "bkpts": res.bkpts,
        "dates": [str(dates[b]) for b in res.bkpts],
        "n_invalid": len(res.invalid_bkpts),
    }

summary = pl.DataFrame(
    [
        bkpt_table("pca", proc_pca),
        bkpt_table("kpca", proc_kpca),
        bkpt_table("dmd", proc_dmd),
    ]
)
summary


## What to try next

- Change `EQ_INDEX` / `N_TICKERS` / date window.
- Raise Graph 2 `max_iter` or lower `threshold` to force more retrain passes.
- Point CLI Graph 2 at a saved `regimes_df` CSV exported from `seed_regimes_from_results(...)`.
